## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-07-19-52-23 +0000,bbc,"'Dana, please take the job!' - Trump asks UFC ...",https://www.bbc.co.uk/news/articles/cvze47p9je...
1,2026-11-07-19-51-44 +0000,bbc,What we know so far after Polish gunman in mil...,https://www.bbc.co.uk/news/articles/cmd7q15drw...
2,2026-11-07-19-50-21 +0000,nyt,Israeli Opposition Leader Tapped to Try to For...,https://www.nytimes.com/2026/11/07/world/middl...
3,2026-11-07-19-50-03 +0000,nypost,NY Attorney General Letitia James with fresh t...,https://nypost.com/2026/11/07/us-news/ny-ag-le...
4,2026-11-07-19-46-07 +0000,nypost,"Troubled ex-NYPD cop, former Marine accused in...",https://nypost.com/2026/11/07/us-news/troubled...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2609/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
3,trump,43
69,new,23
8,house,18
259,illinois,18
149,miller,18


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
184,2026-11-07-02-00-00 +0000,wsj,President Trump has chosen Stephen Miller—an a...,https://www.wsj.com/politics/elections/stephen...,218
245,2026-11-06-22-18-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...,163
304,2026-11-06-20-11-45 +0000,bbc,Trump names Stephen Miller as new White House ...,https://www.bbc.co.uk/news/articles/cm9qzz5x9g...,161
234,2026-11-06-22-53-56 +0000,wapo,Stephen Miller is named new White House chief ...,https://www.washingtonpost.com/politics/2026/1...,136
239,2026-11-06-22-36-00 +0000,wsj,Stephen Miller Is Trump’s New Chief of Staff,https://www.wsj.com/politics/stephen-miller-is...,126


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
184,218,2026-11-07-02-00-00 +0000,wsj,President Trump has chosen Stephen Miller—an a...,https://www.wsj.com/politics/elections/stephen...
232,106,2026-11-06-23-00-02 +0000,chitri,La Shawn K. Ford vs Chad Koppie: How Illinois’...,https://www.chicagotribune.com/2026/11/06/la-s...
66,59,2026-11-07-16-09-00 +0000,wsj,Is AI a white-collar job killer? Strong hiring...,https://www.wsj.com/economy/jobs/is-ai-taking-...
183,55,2026-11-07-02-00-00 +0000,wsj,The wave election that gave Democrats Congress...,https://www.wsj.com/politics/elections/2028-el...
250,46,2026-11-06-22-03-08 +0000,nyt,Susie Wiles Is First High-Level Official to Le...,https://www.nytimes.com/2026/11/06/us/susie-wi...
16,41,2026-11-07-19-09-35 +0000,nypost,Dems gearing up for immigration fight followin...,https://nypost.com/2026/11/07/us-news/dems-gea...
111,41,2026-11-07-11-40-00 +0000,wsj,New Zealand’s ruling conservatives led by a sl...,https://www.wsj.com/world/oceania/new-zealands...
159,40,2026-11-07-07-27-44 +0000,bbc,Meghan's father Thomas Markle under house arre...,https://www.bbc.co.uk/news/articles/cqgkvv8m8p...
176,39,2026-11-07-03-00-00 +0000,wsj,Elon Musk-run companies like SpaceX and Tesla ...,https://www.wsj.com/politics/policy/elon-musk-...
54,37,2026-11-07-16-53-14 +0000,nypost,"Ex-NYPD, ex-Marine maniac arrested in stabbing...",https://nypost.com/2026/11/07/us-news/ex-nypd-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
